In [0]:
%sql
-- GRANT USE CATALOG / USE SCHEMA + SELECT on insurance.gold to an analyst group.
GRANT USE CATALOG ON CATALOG insurance TO `insurance_analysts`;

GRANT USE SCHEMA ON SCHEMA insurance.gold TO `insurance_analysts`;

GRANT SELECT ON SCHEMA insurance.gold TO `insurance_analysts`;

In [0]:
%sql
SHOW GRANTS 
ON SCHEMA insurance.gold;

In [0]:
%sql
-- Add a column mask that hides customers.email / phone from non-privileged groups.

-- Functions used 
CREATE OR REPLACE FUNCTION insurance.gold.mask_email(v STRING)
RETURN CASE 
        WHEN is_account_group_member('pii_readers') THEN v
        ELSE regexp_replace(v, '^.*@', '***@')
       END;

CREATE OR REPLACE FUNCTION insurance.gold.mask_phone(v STRING)
RETURN CASE 
        WHEN is_account_group_member('pii_readers') THEN v
        ELSE 'XXX-XXX-' || right(v, 4)
       END;


-- Actual table altering for masking
ALTER TABLE insurance.gold.dim_customer
ALTER COLUMN email SET MASK insurance.gold.mask_email;

ALTER TABLE insurance.gold.dim_customer
ALTER COLUMN phone SET MASK insurance.gold.mask_phone;

In [0]:
%sql
SELECT email, phone
FROM insurance.gold.dim_customer;

In [0]:
%sql
-- Add a row filter so an agent group only sees rows for their own region.

-- The dim_agent table has region indeed, but dim_customer not really. I assumed the region as the state (the common column)
CREATE OR REPLACE FUNCTION insurance.gold.state_filter(customer_state STRING)
RETURN CASE 
         WHEN is_account_group_member('agents') THEN ( -- if the user is in the agents group
              EXISTS ( -- if there is an agent with the same state as the customer and the user is the agent
                SELECT 1
                FROM insurance.silver.dim_agent da
                WHERE customer_state = da.state AND current_user() = da.email
              )
           )
         ELSE TRUE 
       END;


-- Altering the table
ALTER TABLE insurance.gold.dim_customer
SET ROW FILTER insurance.gold.state_filter ON (state);

In [0]:
%sql
SELECT state 
FROM insurance.gold.dim_customer;